## Data Encoding

1. Nominal/OHE Encoding
2. Label and Ordinal Encoding
3. Target Guided Ordinal Encoding

## Nominal/OHE Encoding

One-hot encoding is a way of turning categorical values into a numeric form that a machine learning model can actually work with, without implying any order between categories. Instead of putting all categories into a single column of numbers, each unique category gets its own column, and for every row only the column matching that row's category is marked `1` while all the others stay `0`.

For example, a "color" column that can take the values red, green, or blue would be split into three separate binary columns like this:

1. Red: [1, 0, 0]
2. Green: [0, 1, 0]
3. Blue: [0, 0, 1]


### Advantages of One-Hot Encoding

1. It doesn't assume any ranking or order among categories, so the model won't wrongly assume that one category is "greater than" or "closer to" another.
2. Works well with algorithms that expect purely numeric input (e.g. linear/logistic regression, SVMs, neural networks).
3. Easy to understand and implement, most libraries (pandas, scikit-learn) support it out of the box.

### Disadvantages of One-Hot Encoding

1. **Curse of dimensionality**: a column with many unique categories creates just as many new columns, which can blow up the size of the dataset.
2. Leads to a **sparse matrix** (mostly 0s), which increases memory usage and can slow down training if not handled efficiently.
3. Doesn't scale well for high-cardinality features like "City" or "Zip Code" with thousands of unique values.

### When to Use / When Not to Use

- **Use it when:** the categorical feature is *nominal* (no inherent order) and has a **small-to-moderate number of unique categories** (e.g. color, gender, department).
- **Avoid it when:** the feature has **high cardinality**, consider label encoding, target/mean encoding, or embeddings instead.
- **Avoid it when:** the categories have a natural **order/rank**, use ordinal/label encoding instead so that order information isn't lost.

> **Rule of thumb:** there's no fixed cutoff, but in practice, once a column has **more than ~10-15 unique categories**, one-hot encoding starts becoming impractical (too many new columns, too sparse) and it's better to switch to label encoding, target/mean encoding, or embeddings. Below that, OHE is generally a safe default.

In [5]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({"color": ["Red", "Green", "Blue", "Green", "Red", "Red", "Green"]})

# Instance
encoder = OneHotEncoder()
encoded = encoder.fit_transform(df[["color"]]).toarray()
encoded

array([[0., 0., 1.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 1., 0.]])

In [6]:
pd.DataFrame(encoded,columns=encoder.get_feature_names_out(["color"]))

,color_Blue,color_Green,color_Red
0,0.0,0.0,1.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,1.0,0.0
4,0.0,0.0,1.0
5,0.0,0.0,1.0
6,0.0,1.0,0.0


In [7]:
encoder.transform(pd.DataFrame({"color": ["Red"]})).toarray()

array([[0., 0., 1.]])

In [8]:
import seaborn as sns
sns.load_dataset('tips')

Matplotlib is building the font cache; this may take a moment.


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


## Label Encoding

Label encoding and ordinal encoding are both ways of converting categorical data into numbers, but they're used in slightly different situations.

With label encoding, every distinct category in a column simply gets swapped for a whole number. There's no fixed rule for how the numbers are assigned, they're typically handed out either alphabetically or based on how often each category shows up. So if we had a "color" column with three possible values (red, green, blue), label encoding could represent it like this:

1. Red: 1
2. Green: 2
3. Blue: 3

### Advantages of Label Encoding

1. Very simple and quick to use.
2. Doesn't add extra columns to your data, so it stays small and easy to handle.
3. Works well when the categories actually have some order (like "Low", "Medium", "High").

### Disadvantages of Label Encoding

1. The model may think the numbers mean something they don't. For example, if Red = 1, Green = 2, Blue = 3, the model might assume Blue is "bigger" or "more important" than Red, which isn't true.
2. Not a good choice when the categories have no natural order, like colors or city names.

### When to Use / When Not to Use

- **Use it when:** the categories have a clear order or ranking (e.g. "Low", "Medium", "High").
- **Avoid it when:** the categories have no order (e.g. colors, names), since the model might get confused and think there's a ranking that doesn't actually exist. In that case, one-hot encoding is usually the safer choice.

In [11]:
from sklearn.preprocessing import LabelEncoder

df = pd.DataFrame({"color": ["Red", "Green", "Blue", "Green", "Red", "Red", "Green"]})
label_encoder = LabelEncoder()
label_encoder.fit_transform(df['color'])

array([2, 1, 0, 1, 2, 2, 1])

In [13]:
label_encoder.transform([['Blue']])

c:\Users\codeanddebug\Desktop\AI-ML-Bootcamp\.venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


array([0])

## Ordinal Encoding

Ordinal encoding is used when a categorical variable has a natural order or ranking to it. Here, each category gets a number based on where it falls in that order, so the numbers themselves carry meaning. For example, if we have a "shirt size" column with four possible values (small, medium, large, extra large), we can represent it using ordinal encoding as follows:

1. Small: 1
2. Medium: 2
3. Large: 3
4. Extra Large: 4

### Advantages of Ordinal Encoding

1. Simple and quick to use, just like label encoding.
2. Doesn't add extra columns to your data, so it stays small and easy to handle.
3. Since the categories actually have a real order, the numbers make sense to the model and don't mislead it.

### Disadvantages of Ordinal Encoding

1. You have to know the correct order of the categories beforehand and set it manually, if you get the order wrong, the model will learn the wrong relationship.
2. The model may assume the gap between each category is equal (e.g. that "Large" is exactly as far from "Medium" as "Medium" is from "Small"), which may not always be true in real life.

### When to Use / When Not to Use

- **Use it when:** the categories have a clear, natural order or ranking (e.g. shirt size, education level, rating like Poor/Average/Good).
- **Avoid it when:** the categories have no natural order (e.g. colors, city names), since forcing an order onto them can confuse the model. In that case, one-hot encoding is usually the safer choice.

In [18]:
from sklearn.preprocessing import OrdinalEncoder

size_df = pd.DataFrame({"shirt_size": ["Medium", "Small", "Large", "Extra Large", "Medium"]})

ordinal_encoder = OrdinalEncoder(categories=[["Small", "Medium", "Large", "Extra Large"]])
ordinal_encoded = ordinal_encoder.fit_transform(size_df[["shirt_size"]])
ordinal_encoded


array([[1.],
       [0.],
       [2.],
       [3.],
       [1.]])

In [ ]:
ordinal_encoder = OrdinalEncoder(
    categories=[["Small", "Medium", "Large", "Extra Large"]],  # tells it the correct order to use
    dtype=int,                            # data type of the encoded numbers (default is float)
    handle_unknown="use_encoded_value",   # what to do if it sees a category it wasn't trained on
    unknown_value=-1,                     # value to give that unknown category (needs handle_unknown set above)
)

## Target Guided Ordinal Encoding

This technique encodes a categorical variable based on how it relates to the target variable (the thing we're actually trying to predict). It's especially handy when a column has a lot of unique categories and we still want to use it as a feature in our model.

Instead of picking numbers arbitrarily, each category is replaced with a number calculated from the target variable, usually the average (or median) target value for that category. Because the numbers now come directly from the target, categories that lead to a higher target value naturally get a higher encoded number. This gives the encoded column a meaningful, ordered relationship with the target, which can help the model make better predictions.

In [20]:
df = pd.DataFrame({
    "job_role": ["Engineer", "Teacher", "Doctor", "Artist", "Engineer", "Doctor", "Teacher", "Artist", "Doctor", "Engineer"],
    "salary": [75000, 45000, 120000, 38000, 80000, 130000, 47000, 40000, 125000, 78000]
})
df

,job_role,salary
0,Engineer,75000
1,Teacher,45000
2,Doctor,120000
3,Artist,38000
4,Engineer,80000
5,Doctor,130000
6,Teacher,47000
7,Artist,40000
8,Doctor,125000
9,Engineer,78000


In [29]:
mean_salary=df.groupby("job_role")["salary"].mean().round(2).to_dict()

In [31]:
df["salary_label"] = df["job_role"].map(mean_salary)
df[["job_role", "salary", "salary_label"]]

,job_role,salary,salary_label
0,Engineer,75000,77666.67
1,Teacher,45000,46000.00
2,Doctor,120000,125000.00
3,Artist,38000,39000.00
4,Engineer,80000,77666.67
5,Doctor,130000,125000.00
6,Teacher,47000,46000.00
7,Artist,40000,39000.00
8,Doctor,125000,125000.00
9,Engineer,78000,77666.67
